# SeaAlert - Notebook 05: ASR Noisy Message Classification with RoBERTa

This notebook demonstrates classification and structured information extraction on **noisy ASR transcripts**.

## Why RoBERTa for ASR?
RoBERTa (Robustly optimized BERT) has been proven to perform better on noisy ASR transcripts because:
- Dynamic masking during pre-training makes it more robust to word variations
- Larger training data includes more diverse text patterns
- Better handling of spelling errors and word substitutions common in ASR output

## Cell 0 - Setup

In [1]:
# Mount Google Drive (Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Install dependencies
!pip install -q transformers torch joblib pandas

# Imports
import os
import re
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import joblib

# Project directory
if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/SeaAlert")
else:
    PROJECT_DIR = Path(".").resolve().parent

RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"
DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Add src to path
sys.path.insert(0, str(PROJECT_DIR / "src"))

# Labels
LABELS = ["Routine", "Safety", "Urgency", "Distress"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

print(f"PROJECT_DIR: {PROJECT_DIR}")

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/SeaAlert


## Cell 1 - Load RoBERTa Model (Best for Noisy ASR)

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# MODEL SELECTION - Using RoBERTa (proven better for noisy ASR)
# ============================================================
USE_MODEL = "roberta"  # RoBERTa is more robust to ASR noise
MODEL_NAME = "roberta-base"

# Try to load fine-tuned RoBERTa from local models directory
ROBERTA_MODEL_PATH = MODELS_DIR / "best_roberta"
TRANSFORMER_MODEL_PATH = MODELS_DIR / "best_transformer"

# Priority: best_roberta > best_transformer > download from HuggingFace
model_loaded = False

if ROBERTA_MODEL_PATH.exists():
    print(f"Loading fine-tuned RoBERTa from: {ROBERTA_MODEL_PATH}")
    tf_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_PATH)
    tf_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL_PATH)
    model_loaded = True
    MODEL_NAME = "roberta-base (fine-tuned)"
elif TRANSFORMER_MODEL_PATH.exists():
    print(f"Loading transformer model from: {TRANSFORMER_MODEL_PATH}")
    tf_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_PATH)
    tf_model = AutoModelForSequenceClassification.from_pretrained(TRANSFORMER_MODEL_PATH)
    model_loaded = True
    MODEL_NAME = "transformer (fine-tuned)"
else:
    # Download RoBERTa base and create classifier head
    print(f"No fine-tuned model found. Loading {MODEL_NAME} from HuggingFace...")
    tf_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
    tf_model = AutoModelForSequenceClassification.from_pretrained(
        'roberta-base',
        num_labels=4,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )
    model_loaded = True
    print("Note: Using base RoBERTa without fine-tuning. Results may vary.")

# Move to device
if model_loaded:
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    tf_model = tf_model.to(device)
    tf_model.eval()
    print(f"\nModel loaded successfully!")
    print(f"   Model: {MODEL_NAME}")
    print(f"   Device: {device}")
else:
    tf_model = None
    tf_tokenizer = None
    print("No model available")

Loading transformer model from: /content/drive/MyDrive/SeaAlert/models/best_transformer

Model loaded successfully!
   Model: transformer (fine-tuned)
   Device: cuda


## Cell 2 - Classification Functions

In [3]:
def classify_roberta(text, model, tokenizer):
    """Classify text with RoBERTa model."""
    device = next(model.parameters()).device

    encoding = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors='pt'
    )
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    pred_idx = np.argmax(probs)
    pred_label = ID2LABEL[pred_idx]
    confidence = float(probs[pred_idx])

    return pred_label, confidence, probs

## Cell 3 - Information Extraction (Regex)

In [4]:
# Regex patterns for extraction
EXTRACTION_PATTERNS = {
    "vessel": [
        r"(?:vessel|ship|boat|yacht|mv|m/v|motor vessel)\s+['\"]?([A-Za-z][A-Za-z0-9\s\-']+)['\"]?",
        r"this is\s+(?:the\s+)?(?:fishing vessel\s+|vessel\s+)?['\"]?([A-Za-z][A-Za-z0-9\s\-']+)['\"]?(?:\s*,|\s+call)",
    ],
    "call_sign": [
        r"call\s*sign\s*:?\s*([A-Z0-9]{4,8})",
        r"\b([A-Z]{2,3}\d{4})\b",
    ],
    "mmsi": [
        r"MMSI\s*:?\s*(\d{9})",
        r"\b(\d{9})\b",
    ],
    "position": [
        r"position\s*:?\s*([\d]+\s*degrees?\s*[\d.]+\s*(?:minutes?)?\s*[NS][\s,]+[\d]+\s*degrees?\s*[\d.]+\s*(?:minutes?)?\s*[EW])",
        r"(\d{1,3}\s*degrees?\s*\d{1,2}(?:\.\d+)?\s*minutes?\s*[NS][\s,]+\d{1,3}\s*degrees?\s*\d{1,2}(?:\.\d+)?\s*minutes?\s*[EW])",
        r"at\s+(?:position\s+)?(\d{1,3}\s*degrees?\s*\d{1,2}(?:\.\d+)?\s*minutes?\s*[nN]orth[\s,]+\d{1,3}\s*degrees?\s*\d{1,2}(?:\.\d+)?\s*minutes?\s*[wW]est)",
        r"(\d{1,3}[°\.]\d{1,2}['']?\s*[NS][\s,]+\d{1,3}[°\.]\d{1,2}['']?\s*[EW])",
    ],
    "pob": [
        r"(?:there are\s+)?(\d{1,3})\s*(?:persons?|pob|people|souls?)\s*(?:on\s*board|aboard)?",
        r"(?:persons?|pob|people|souls?)\s*(?:on\s*board|aboard)?\s*:?\s*(\d{1,3})",
        r"POB\s*:?\s*(\d{1,3})",
    ],
    "nature": [
        r"(fire|explosion|flooding|water ingress|taking on water|hull breach|engine failure|medical emergency|injury|person overboard|man overboard|collision|grounding|grounded|sinking|capsized|disabled|adrift|steering failure|complete loss of power|allergic reaction|tow)",
    ],
    "injury": [
        r"(\d+)\s*(?:injured|casualties|with injuries)",
        r"(no injuries)",
        r"(injur(?:y|ed|ies)|casualt(?:y|ies)|burn|fracture|unconscious|bleeding|medical|allergic)",
    ],
    "weather": [
        r"(?:weather|conditions|sea state)\s*(?:are|is)?\s*:?\s*([^.]+(?:winds?|seas?|swells?|knots?|meters?)[^.]*)",
        r"(clear|calm|rough|moderate|deteriorating)[^.]*(?:wind|sea|swell|knot|meter)",
    ],
    "assistance": [
        r"(?:require|request|need|requesting)(?:s|ing)?\s+(?:immediate\s+)?(towing|tow|evacuation|medical assistance|assistance|rescue|helicopter|lifeboat|coast guard)",
    ],
}

def extract_with_regex(text):
    """Extract structured fields using regex."""
    result = {}

    for field, patterns in EXTRACTION_PATTERNS.items():
        result[field] = None
        for pattern in patterns:
            try:
                match = re.search(pattern, text, re.IGNORECASE)
                if match:
                    value = match.group(1) if match.lastindex else match.group(0)
                    value = value.strip().strip("'\"")

                    # Convert POB to int
                    if field == "pob":
                        try:
                            value = int(value)
                        except:
                            pass

                    result[field] = value
                    break
            except:
                continue

    # Detect codeword
    text_upper = text.upper()
    if "MAYDAY" in text_upper:
        result["codeword"] = "MAYDAY"
    elif "PAN PAN" in text_upper or "PAN-PAN" in text_upper:
        result["codeword"] = "PAN PAN"
    elif "SECURITE" in text_upper:
        result["codeword"] = "SECURITE"
    else:
        result["codeword"] = None

    return result

## Cell 4 - Load Dataset and Select FORMAL Style Message

In [5]:
# ============================================================
# LOAD DATASET AND FIND FORMAL STYLE MESSAGE
# ============================================================

# Try to load the ASR dataset first, fallback to base dataset
ASR_DATASET_PATH = DATA_DIR / "03seaalert_with_asr.csv"
BASE_DATASET_PATH = DATA_DIR / "02seaalert.csv"

HAS_ASR_DATA = False

if ASR_DATASET_PATH.exists():
    df = pd.read_csv(ASR_DATASET_PATH)
    HAS_ASR_DATA = "asr_high" in df.columns
    print(f"Loaded ASR dataset: {len(df)} samples")
elif BASE_DATASET_PATH.exists():
    df = pd.read_csv(BASE_DATASET_PATH)
    print(f"Loaded base dataset: {len(df)} samples")
else:
    raise FileNotFoundError("Dataset not found!")

print(f"Columns: {list(df.columns)}")
print(f"\nStyle distribution:")
print(df['style'].value_counts())

Loaded ASR dataset: 1872 samples
Columns: ['idx', 'text', 'label', 'style', 'scenario_type', 'has_codeword', 'codeword', 'text_masked', 'vessel', 'call_sign', 'mmsi', 'location', 'weather', 'pob', 'nature', 'injury', 'asr_low', 'asr_med', 'asr_high']

Style distribution:
style
formal         624
third_party    624
informal       624
Name: count, dtype: int64


In [6]:
# ============================================================
# SELECT A FORMAL STYLE MESSAGE WITH RICH DATA
# ============================================================

# Filter for formal style messages
formal_df = df[df['style'] == 'formal'].copy()
print(f"Found {len(formal_df)} formal style messages")

# Prefer Distress or Urgency with rich metadata
formal_rich = formal_df[
    (formal_df['label'].isin(['Distress', 'Urgency'])) &
    (formal_df['call_sign'].notna()) &
    (formal_df['mmsi'].notna()) &
    (formal_df['location'].notna()) &
    (formal_df['pob'].notna())
]
print(f"Found {len(formal_rich)} formal messages with rich metadata")

# Select a sample
if len(formal_rich) > 0:
    sample = formal_rich.sample(1, random_state=42).iloc[0]
elif len(formal_df) > 0:
    sample = formal_df.sample(1, random_state=42).iloc[0]
else:
    sample = df.sample(1, random_state=42).iloc[0]

# Get the original text
ORIGINAL_TEXT = sample['text']
TRUE_LABEL = sample['label']
STYLE = sample.get('style', 'unknown')
SCENARIO = sample.get('scenario_type', 'unknown')

# Store dataset metadata for later
DATASET_METADATA = {
    'vessel': sample.get('vessel'),
    'call_sign': sample.get('call_sign'),
    'mmsi': sample.get('mmsi'),
    'location': sample.get('location'),
    'weather': sample.get('weather'),
    'pob': sample.get('pob'),
    'nature': sample.get('nature'),
    'injury': sample.get('injury'),
    'codeword': sample.get('codeword'),
    'scenario_type': sample.get('scenario_type'),
}

print(f"\nSelected message:")
print(f"   Label: {TRUE_LABEL}")
print(f"   Style: {STYLE}")
print(f"   Scenario: {SCENARIO}")

Found 624 formal style messages
Found 221 formal messages with rich metadata

Selected message:
   Label: Distress
   Style: formal
   Scenario: tow_request


In [7]:
# ============================================================
# GET ASR TEXT (Real or Simulated)
# ============================================================

def simulate_asr_noise(text, error_rate=0.15):
    """Simulate ASR transcription errors on clean text."""
    substitutions = {
        'degrees': ['degree', 'degrees'],
        'minutes': ['minute', 'minits'],
        'vessel': ['vessel', 'vesel'],
        'persons': ['persons', 'person'],
        'injured': ['injured', 'injerd'],
        'assistance': ['assistance', 'asistance'],
        'mayday': ['mayday', 'may day'],
        'position': ['position', 'posision'],
        'grounded': ['grounded', 'grownded'],
        'failure': ['failure', 'faliure'],
        'hull': ['hull', 'hall'],
        'breach': ['breach', 'breech'],
        'tow': ['tow', 'toe'],
        'weather': ['weather', 'wether'],
    }

    noisy = text.lower().replace('.', ' ').replace(',', ' ')
    words = noisy.split()
    noisy_words = []

    for word in words:
        if word in substitutions and random.random() < error_rate:
            noisy_words.append(random.choice(substitutions[word]))
        else:
            noisy_words.append(word)

    return re.sub(r'\s+', ' ', ' '.join(noisy_words)).strip()

# Check if we have actual ASR data
if HAS_ASR_DATA and 'asr_high' in sample.index and pd.notna(sample.get('asr_high')):
    ASR_TEXT = sample['asr_high']
    ASR_SOURCE = "Real ASR (from audio pipeline)"
else:
    random.seed(42)
    ASR_TEXT = simulate_asr_noise(ORIGINAL_TEXT, error_rate=0.20)
    ASR_SOURCE = "Simulated ASR"

print(f"ASR Source: {ASR_SOURCE}")

ASR Source: Real ASR (from audio pipeline)


## Cell 5 - Display Original vs ASR Text

In [8]:
# Display original vs ASR text
print("="*80)
print("ORIGINAL MESSAGE")
print("="*80)
print(ORIGINAL_TEXT)
print()
print("="*80)
print(f"ASR TRANSCRIPT ({ASR_SOURCE})")
print("="*80)
print(ASR_TEXT)
print()

from difflib import SequenceMatcher
similarity = SequenceMatcher(None, ORIGINAL_TEXT.lower(), ASR_TEXT.lower()).ratio()
print(f"Text Similarity: {similarity:.1%}")
print(f"Text Degradation: {(1-similarity):.1%}")

ORIGINAL MESSAGE
MAYDAY, MAYDAY, MAYDAY. This is the fishing vessel Ocean Explorer, call sign WXYZ123, MMSI 123456789. We are adrift, approximately 15 nautical miles east of Cape Point, at position 34 degrees 12 minutes South, 18 degrees 29 minutes East. The vessel's engine has failed, and we are currently taking on water. Weather conditions are worsening with 4-meter swells and visibility reduced to 2 nautical miles. There are 6 persons on board. We require immediate assistance for towing. Repeat, we are requesting a tow. Over.

ASR TRANSCRIPT (Real ASR (from audio pipeline))
maybe, maybe, maybe. This is the Fishing Vessel Oceanate Spoiler. Paul  Signed to be its Ryzen 123 MMSI 120 3 million 456000 7809. The Area Drift  approximately 15 nautical miles east of Cape Point, a position 34 degrees 12 minutes south,  18 degrees 29 minutes east. The Vessel's engine has failed and we are currently taking on water.  Whether conditions are a worse name before need as well as invisibility we cho

## Cell 6 - Classify with RoBERTa

In [9]:
# Classify both texts
orig_label, orig_confidence, orig_probs = None, None, None
asr_label, asr_confidence, asr_probs = None, None, None

if tf_model is not None:
    orig_label, orig_confidence, orig_probs = classify_roberta(ORIGINAL_TEXT, tf_model, tf_tokenizer)
    asr_label, asr_confidence, asr_probs = classify_roberta(ASR_TEXT, tf_model, tf_tokenizer)

    print(f"Original: {orig_label} ({orig_confidence:.1%})")
    print(f"ASR:      {asr_label} ({asr_confidence:.1%})")
    print(f"Truth:    {TRUE_LABEL}")

Original: Distress (98.0%)
ASR:      Distress (85.1%)
Truth:    Distress


## Cell 7 - Extract Information

In [10]:
# Extract information
orig_extraction = extract_with_regex(ORIGINAL_TEXT)
asr_extraction = extract_with_regex(ASR_TEXT)

## Cell 8 - Simple Clean Output (Main Result)

In [11]:
# ============================================================
# SIMPLE CLEAN OUTPUT - Main Result
# ============================================================
# This is the main output showing the ASR message classification
# and all extracted parameters in a simple, clear format

print()
print("#" * 80)
print("#" + " " * 78 + "#")
print("#" + "SEAALERT - MESSAGE ANALYSIS RESULT".center(78) + "#")
print("#" + " " * 78 + "#")
print("#" * 80)
print()

# Message
print("MESSAGE (ASR Input):")
print("-" * 80)
print(ASR_TEXT)
print()

# Classification
print("=" * 80)
print("CLASSIFICATION")
print("=" * 80)
print(f"Label:              {asr_label}")
print(f"Confidence:         {asr_confidence:.1%}")
print(f"Ground Truth:       {TRUE_LABEL}")
print(f"Correct:            {'YES' if asr_label == TRUE_LABEL else 'NO'}")
print()

# Extracted Parameters
print("=" * 80)
print("EXTRACTED PARAMETERS")
print("=" * 80)
print(f"Codeword:           {asr_extraction.get('codeword') or 'Not detected'}")
print(f"Vessel:             {asr_extraction.get('vessel') or 'Not detected'}")
print(f"Call Sign:          {asr_extraction.get('call_sign') or 'Not detected'}")
print(f"MMSI:               {asr_extraction.get('mmsi') or 'Not detected'}")
print(f"Position:           {asr_extraction.get('position') or 'Not detected'}")
print(f"POB:                {asr_extraction.get('pob') or 'Not detected'}")
print(f"Nature/Scenario:    {asr_extraction.get('nature') or 'Not detected'}")
print(f"Injury:             {asr_extraction.get('injury') or 'Not detected'}")
print(f"Weather:            {asr_extraction.get('weather') or 'Not detected'}")
print(f"Assistance:         {asr_extraction.get('assistance') or 'Not detected'}")
print()

# Comparison with Ground Truth (from dataset)
print("=" * 80)
print("GROUND TRUTH (from Dataset)")
print("=" * 80)
print(f"Codeword:           {DATASET_METADATA.get('codeword') or 'N/A'}")
print(f"Vessel:             {DATASET_METADATA.get('vessel') or 'N/A'}")
print(f"Call Sign:          {DATASET_METADATA.get('call_sign') or 'N/A'}")
print(f"MMSI:               {DATASET_METADATA.get('mmsi') or 'N/A'}")
print(f"Location:           {DATASET_METADATA.get('location') or 'N/A'}")
print(f"POB:                {DATASET_METADATA.get('pob') or 'N/A'}")
print(f"Nature/Scenario:    {DATASET_METADATA.get('nature') or 'N/A'}")
print(f"Scenario Type:      {DATASET_METADATA.get('scenario_type') or 'N/A'}")
print(f"Injury:             {DATASET_METADATA.get('injury') or 'N/A'}")
print(f"Weather:            {DATASET_METADATA.get('weather') or 'N/A'}")
print()

# Summary line
print("=" * 80)
print(f"ASR Degradation: {(1-similarity):.1%} | Model: {MODEL_NAME}")
print("=" * 80)


################################################################################
#                                                                              #
#                      SEAALERT - MESSAGE ANALYSIS RESULT                      #
#                                                                              #
################################################################################

MESSAGE (ASR Input):
--------------------------------------------------------------------------------
maybe, maybe, maybe. This is the Fishing Vessel Oceanate Spoiler. Paul  Signed to be its Ryzen 123 MMSI 120 3 million 456000 7809. The Area Drift  approximately 15 nautical miles east of Cape Point, a position 34 degrees 12 minutes south,  18 degrees 29 minutes east. The Vessel's engine has failed and we are currently taking on water.  Whether conditions are a worse name before need as well as invisibility we choose to  T-Nautical miles. There are six persons on board. You require immed

## Cell 9 - Detailed Report (Optional)

In [12]:
# ============================================================
# DETAILED FORMATTED REPORT (Optional)
# ============================================================

def generate_seaalert_report(label, confidence, probs, extraction, true_label, model_name, asr_source):
    """Generate formatted report."""
    severity_info = {
        "Distress": {"icon": "[!!!]", "priority": "CRITICAL"},
        "Urgency": {"icon": "[!!]", "priority": "HIGH"},
        "Safety": {"icon": "[!]", "priority": "MEDIUM"},
        "Routine": {"icon": "[i]", "priority": "LOW"}
    }
    info = severity_info.get(label, {"icon": "[?]", "priority": "UNKNOWN"})

    lines = []
    w = 78

    lines.append("+" + "="*w + "+")
    lines.append("|" + "SEAALERT ANALYSIS REPORT".center(w) + "|")
    lines.append("+" + "="*w + "+")
    lines.append("|" + f"  Severity: {info['icon']} {label.upper()} ({info['priority']})".ljust(w) + "|")
    lines.append("|" + f"  Confidence: {confidence:.1%}".ljust(w) + "|")
    lines.append("|" + f"  Ground Truth: {true_label}".ljust(w) + "|")
    lines.append("|" + f"  Status: {'CORRECT' if label == true_label else 'INCORRECT'}".ljust(w) + "|")
    lines.append("+" + "-"*w + "+")

    for lbl, prob in zip(LABELS, probs):
        bar = "#" * int(prob * 30) + "." * (30 - int(prob * 30))
        lines.append("|" + f"    {lbl:<10}: [{bar}] {prob:>6.1%}".ljust(w) + "|")

    lines.append("+" + "="*w + "+")
    return "\n".join(lines)

if asr_label:
    print(generate_seaalert_report(asr_label, asr_confidence, asr_probs, asr_extraction, TRUE_LABEL, MODEL_NAME, ASR_SOURCE))

+==============================================================================+
|                           SEAALERT ANALYSIS REPORT                           |
+==============================================================================+
|  Severity: [!!!] DISTRESS (CRITICAL)                                         |
|  Confidence: 85.1%                                                           |
|  Ground Truth: Distress                                                      |
|  Status: CORRECT                                                             |
+------------------------------------------------------------------------------+
|    Routine   : [..............................]   2.0%                       |
|    Safety    : [#.............................]   6.0%                       |
|    Urgency   : [##............................]   7.0%                       |
|    Distress  : [#########################.....]  85.1%                       |
+===========================

## Cell 10 - Save Report

In [13]:
# Save comprehensive report as JSON
if asr_label is not None:
    demo_report = {
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "model": MODEL_NAME,
            "asr_source": ASR_SOURCE
        },
        "input": {
            "original_text": ORIGINAL_TEXT,
            "asr_text": ASR_TEXT,
            "style": STYLE,
            "similarity": float(similarity)
        },
        "classification": {
            "predicted": asr_label,
            "confidence": float(asr_confidence),
            "ground_truth": TRUE_LABEL,
            "correct": asr_label == TRUE_LABEL
        },
        "extraction": asr_extraction,
        "ground_truth_metadata": DATASET_METADATA
    }

    REPORT_PATH = RESULTS_DIR / "demo_asr_roberta_report.json"
    with open(REPORT_PATH, 'w') as f:
        json.dump(demo_report, f, indent=2, default=str)

    print(f"Report saved to: {REPORT_PATH}")

Report saved to: /content/drive/MyDrive/SeaAlert/results/demo_asr_roberta_report.json
